# 06 · 官方算例展示与黄金比对

本 notebook 有三重用途, 按顺序看即可了解整个工作台:

1. **工作流演示** — 从官方输入卡到可执行文件、再到统计与绘图,
   展示 astra-notebook 的完整数据流 (文件 -> 读取(SI) -> 计算 ->
   展示/导出);
2. **结果展示** — 每个算例运行后直接给出统计表、演化图、相空间图,
   与 02/03/04/05 号 notebook 的专职视图一一对应;
3. **黄金比对** — 一键复现全部 9 个官方算例, 与归档的 golden 输出
   逐项比对 (rel < 0.5% 判 OK), 验证本机环境与后端正确性。

工作台地图 (与原程序一一对应):

| # | notebook | 对应原程序 | 本 notebook 对应展示 |
|---|----------|-----------|----------------------|
| 01 | generator | generator | Manual_Example 的 generator.in |
| 02 | astra | astra | 输入卡打印 + 输出清单 |
| 03 | postpro | postpro | 统计表 + 相空间图 + slice 图 |
| 04 | lineplot | lineplot | 发射度/能量演化图 |
| 05 | fieldplot | fieldplot | 等离子体密度剖面 |
| 06 | examples | — | 本 notebook (展示 + 验证) |

注意: Plasma_Example_2 需先下载 plasma_user_laser.zip 并解压到
examples/Plasma_Example_2/ (65MB laser.dat 不进 git)。

In [ ]:
%run _bootstrap.py

## 算例目录 (DESY 官方 9 例)

| 算例 | 物理内容 |
|------|----------|
| Manual_Example | 手册主算例: 光阴极枪 + 螺线管, 发射度补偿 |
| Aperture | 圆孔/刮板/堵块等孔径 |
| Wake | 尾场 (TESLA 模块) |
| Cavity_Example | 1D 腔/TWS/3D 场图/TDS 行波结构 |
| Curved_Cathode_Example | 弯曲阴极 |
| 90deg_bend_Example | 90° 弯转 (3D 二极场) |
| Plasma_Example_1 | 束驱动等离子体尾场 |
| Plasma_Example_2 | 激光驱动等离子体尾场 (需本地 laser.dat) |
| WHPS_LINAC | WHPS 直线加速器 (大文件输入, 见 examples/WHPS_LINAC) |


In [ ]:
# ===== 共享: 9 算例定义 + 运行/比对机制 =====
import shutil, json
from pathlib import Path
from astra_tools.run import run_program
from astra_tools.io.astra_emit import parse_output_file
from astra_tools.io.field_map import fix_laser_map_header

EXAMPLES_DIR = PROJECT_ROOT / "examples"
GOLDEN_EXPECTED = json.loads((EXAMPLES_DIR / "golden_expected.json").read_text())

# 每个算例: 输入文件清单 / 运行步骤 / 黄金比对目标
EXAMPLES = {
    "Manual_Example": dict(
        copy=["generator.in", "Example.in", "3_cell_L-Band.dat", "Solenoid.dat"],
        patch={"Example.in": (
            "/Users/yuxinwu/astra_notebook/simulation_files/Example.ini",
            "Example.ini")},
        steps=[("generator", "generator.in"), ("astra", "Example.in")],
        golden_xemit=EXAMPLES_DIR / "Manual_Example/Example.Xemit.001"),
    "Aperture": dict(
        copy=["astra.in", "aperture.in", "Geometry.dat", "test.ini"],
        steps=[("astra", "astra.in")],
        golden_xemit=EXAMPLES_DIR / "Aperture/golden/astra.Xemit.001"),
    "Wake": dict(
        copy=["Wake.in", "test.ini", "TESLA_MODULE_WAKE_TAYLOR.dat", "test.dat"],
        src_dir="Wake/Wake_Files",
        steps=[("astra", "Wake.in")],
        golden_xemit=EXAMPLES_DIR / "Wake/golden/Wake.Xemit.001"),
    "Cavity_Example": dict(
        copy=["generator.in", "astra.in", "TWS_Sband.dat", "3_cell_L-Band.dat",
              "dcfield.dat", "3D_test.bx", "3D_test.by", "3D_test.bz"],
        steps=[("generator", "generator.in"), ("astra", "astra.in")],
        golden_xemit=EXAMPLES_DIR / "Cavity_Example/golden/astra.Xemit.001"),
    "Curved_Cathode_Example": dict(
        copy=["generator.in", "astra.in", "Contour.dat", "efld.dat"],
        steps=[("generator", "generator.in"), ("astra", "astra.in")],
        golden_xemit=EXAMPLES_DIR / "Curved_Cathode_Example/golden/astra.Xemit.001"),
    "90deg_bend_Example": dict(
        copy=["Section1.in", "Section2.in", "test.ini",
              "3D_Dipole.bx", "3D_Dipole.by", "3D_Dipole.bz"],
        patch={"Section2.in": ("Section1_n.0100.001", "Section1.0100.001")},
        steps=[("astra", "Section1.in"), ("astra", "Section2.in")],
        golden_xemit=EXAMPLES_DIR / "90deg_bend_Example/golden/Section2.Log.001",
        compare_mode="log"),
    "Plasma_Example_1": dict(
        copy=["phsp.in", "plasma.in", "PLASMA_flattop.txt"],
        steps=[("generator", "phsp.in"), ("astra", "plasma.in")],
        golden_xemit=EXAMPLES_DIR / "Plasma_Example_1/golden/plasma.Xemit.001"),
    "Plasma_Example_2": dict(
        copy=["phsp.in", "plasma.in", "PLASMA_flattop.txt", "laser.dat"],
        laser_fix=True,
        steps=[("generator", "phsp.in"), ("astra", "plasma.in")],
        golden_xemit=EXAMPLES_DIR / "Plasma_Example_2/golden/plasma.Xemit.001"),
}


def run_example(name):
    """把官方算例输入复制到工作目录并运行 generator/astra。"""
    spec = EXAMPLES[name]
    src = EXAMPLES_DIR / spec.get("src_dir", name)
    work = SIM_DIR / name
    work.mkdir(parents=True, exist_ok=True)
    for f in spec["copy"]:
        s = src / f
        if not s.exists():
            raise FileNotFoundError(
                "%s 缺失: %s" % (f, s))
        shutil.copy2(s, work / f)
    for deck, (old, new) in spec.get("patch", {}).items():
        p = work / deck
        p.write_text(p.read_text().replace(old, new))
    if spec.get("laser_fix"):
        fix_laser_map_header(work / "laser.dat")
        print("  laser.dat 图头计数已转为整数形式")
    for kind, deck in spec["steps"]:
        exe = GENERATOR_EXE if kind == "generator" else ASTRA_EXE
        if exe is None:
            raise RuntimeError("未找到 %s 可执行文件" % kind)
        run_program(exe, work, input_file=deck)
    return work


def compare_xemit(name, work):
    """新运行 vs 归档 golden 的末行比对 (rel < 0.5% 判 OK)。"""
    spec = EXAMPLES[name]
    golden = spec["golden_xemit"]
    new_file = work / golden.name
    if not new_file.exists():
        print("  (无 %s 输出, 跳过比对)" % golden.name)
        return
    if spec.get("compare_mode") == "log":
        # Log 文件比对: 都包含成功结束标记即 OK
        ok_new = "finished" in new_file.read_text()
        ok_ref = "finished" in golden.read_text()
        print("  末行比对 (new vs golden, log):")
        print("    finished        %-10s %-10s %s"
              % (ok_new, ok_ref, "OK" if ok_new and ok_ref else "MISMATCH"))
        return
    new = parse_output_file(new_file)
    ref = parse_output_file(golden)
    print("  末行比对 (new vs golden):")
    for key in ("norm_emit_x", "sigma_x", "mean_z"):
        a = float(__import__("numpy").asarray(new[key])[-1])
        b = float(__import__("numpy").asarray(ref[key])[-1])
        rel = abs(a - b) / abs(b) * 100
        print("    %-14s %-10.6g %-10.6g rel=%.4f%% %s"
              % (key, a, b, rel, "OK" if rel < 0.5 else "MISMATCH"))


## 展示一: Manual_Example 完整工作流

下面一个单元走完整个数据流: 运行 -> 输入卡 -> 输出清单 ->
束团统计 (与 ASTRA Xemit 交叉验证 <0.02%) -> 演化图/相空间图 ->
数据导出。对应 notebook 01+02+03+04+05+07 的全部环节。

In [ ]:
EXAMPLE = "Manual_Example"
work = run_example(EXAMPLE)

# ---- 1) 输入卡 (astra.in, 即 02 号 notebook 生成的产物) ----
print("===== 输入卡 astra.in (节选) =====")
print("".join((work / "Example.in").read_text().splitlines(True)[:20]))

# ---- 2) 输出文件清单 (03 号 notebook 的 discover_outputs) ----
outs = sorted(f.name for f in work.glob("Example.*"))
print("===== 输出文件 (%d 个) =====" % len(outs))
print("  " + ", ".join(outs))

# ---- 3) 束团统计: 控制台 + HTML 面板 (04 号 notebook) ----
from astra_tools.io import read_distribution
from astra_tools.analysis.statistics import compute_statistics, print_statistics
from astra_tools.widgets.panels import stats_table_html
dist = read_distribution(work / "Example.0150.001")
stats = compute_statistics(dist)
print_statistics(stats, title="Manual_Example @ z=1.5 m")
stats_table_html(stats)

# ---- 4) 演化图 (05 号 notebook) 与相空间/slice 图 (04/07) ----
from astra_tools.io.astra_emit import read_emit_files
from astra_tools.plot.emit_plots import plot_emit_dashboard
from astra_tools.plot.phase_space import plot_phase_space
from astra_tools.analysis.slices import compute_slice_analysis
from astra_tools.plot.slice_plots import plot_slice_dashboard
emit = read_emit_files(str(work / "Example"))
plot_emit_dashboard(emit)
plot_phase_space(dist, plane="x", show_ellipse=True)
sa = compute_slice_analysis(dist, n_slices=20)
plot_slice_dashboard(sa)

# ---- 5) 数据导出 (任何 notebook 都可导出 CSV/npz) ----
from astra_tools.export import export_statistics, export_emit
out_dir = work / "export"
print("导出目录:", out_dir)
print("  统计表:", export_statistics(stats, out_dir).name)
print("  演化数据:", ", ".join(p.name for p in export_emit(emit, out_dir).values()))

# ---- 6) 黄金比对 (本 notebook 第 3 重用途) ----
compare_xemit(EXAMPLE, work)


## 展示二: Plasma_Example_1 (不同物理: 等离子体尾场)

同样的数据流, 换一个物理场景: 展示 fieldplot 侧的等离子体密度剖面
与能量演化。

In [ ]:
EXAMPLE = "Plasma_Example_1"
work = run_example(EXAMPLE)

from astra_tools.io.astra_emit import read_emit_files
from astra_tools.plot.emit_plots import plot_energy_evolution, plot_emittance_evolution
from astra_tools.plot.advanced_plots import plot_plasma_profile
emit = read_emit_files(str(work / "plasma"))
plot_energy_evolution(emit)
plot_emittance_evolution(emit)
plot_plasma_profile(EXAMPLES_DIR / "Plasma_Example_1/PLASMA_flattop.txt",
                   peak_density_cm3=1e17)
compare_xemit(EXAMPLE, work)


## 一键复现全部 9 个算例 + 黄金比对

约 1 分钟。任何一行 FAILED 或 MISMATCH 都说明本机环境或后端有
问题 (完整回归测试见 test/ 与 docs/dev_manual/test_plan.md)。

In [ ]:
for name in EXAMPLES:
    print("=" * 60)
    print(name)
    try:
        work = run_example(name)
        compare_xemit(name, work)
    except Exception as e:
        print("  FAILED:", e)
    print()
